In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/xpu

In [ ]:
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

In [ ]:
pip install opencv-python ultralytics opencv-contrib-python numpy Pillow ipython

In [ ]:
import torch
print(torch.__version__)
print(torch.xpu.is_available())

2.11.0+cu130
True


In [ ]:
import torch
x = torch.ones(3, device="xpu")
print(x * 2)

tensor([2., 2., 2.])


In [7]:
import cv2
from ultralytics import YOLO

model = YOLO("models/yolo26s.pt")
face_model = YOLO("models/yolov10m-face.pt")

photo = cv2.imread("p.img01.jpg")
base_img = model(photo)
base_result = base_img[0].plot()

face_img = face_model(photo)
face_result = face_img[0].plot(img=base_result)

# result
# cv2.imwrite("exported12n.jpg", result[0].plot())

cv2.imshow("RESULT", face_result)
cv2.waitKey(0)
cv2.destroyAllWindows()
# cv2.moveWindow("MY PHOTO", 1800, 200)

# cv2.waitKey(5000)
# cv2.destroyAllWindows()


0: 384x640 4 persons, 2 bottles, 3 cups, 2 chairs, 1 bed, 1 dining table, 1 remote, 324.7ms
Speed: 11.8ms preprocess, 324.7ms inference, 21.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 faces, 388.5ms
Speed: 3.8ms preprocess, 388.5ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)


In [ ]:
# FACE AND OBJECT DETECTION

cap = cv2.VideoCapture(0)
model = YOLO("models/yolo26s.pt")
face_model = YOLO("models/yolov10m-face.pt")

while cv2.waitKey(1) != ord("x"):
    _, frame = cap.read()

    base_img = model(frame, verbose=False)
    base_result = base_img[0].plot()
    
    face_img = face_model(base_result, verbose=False)
    face_result = face_img[0].plot()
    
    cv2.imshow("RESULT", face_result)
    
cv2.waitKey(0)
cap.release()
cv2.destroyAllWindows()

# Face Recognization

In [1]:
import os
import cv2
import numpy as np
from ultralytics import YOLO
from IPython.display import display
from PIL import Image

In [ ]:
# Single Person
folder = "faces/Ridoy"
files = os.listdir(folder)

face_model = YOLO("models/yolov10m-face.pt")
face_recognizer = cv2.face.LBPHFaceRecognizer_create()
faces = []
labels = []

for file in files:
    photo = cv2.imread(folder + "/" + file)
    face_img = face_model(photo, device="xpu", verbose=False)
    face_result = face_img[0].plot()

    left, top, right, bottom = face_img[0].boxes.xyxy[0].int()
    face = photo[top:bottom, left:right]
    face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
    face = cv2.resize(face, (200, 200))

    faces.append(face)
    labels.append(0)
    
face_recognizer.train(faces, np.array(labels))
face_recognizer.write("face_recognizer.yml")
    
# Image.fromarray(face[:, :, ::-1])
Image.fromarray(faces[0])

In [58]:
# Multiple Person

face_model = YOLO("models/yolov10m-face.pt")
face_recognizer = cv2.face.LBPHFaceRecognizer_create()

faces = []
labels = []

folder = "faces/Ridoy"
folders = {0: "faces/Ridoy"}

for label, folder in folders.items():
    files = os.listdir(folder)
    
    for file in files:
        photo = cv2.imread(folder + "/" + file)
        face_img = face_model(photo, device="xpu", verbose=False)
        face_result = face_img[0].plot()
    
        left, top, right, bottom = face_img[0].boxes.xyxy[0].int()
        face = photo[top:bottom, left:right]
        face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
        face = cv2.resize(face, (200, 200))
    
        faces.append(face)
        labels.append(label)
    
face_recognizer.train(faces, np.array(labels))
face_recognizer.write("face_recognizer.yml")
print("Complete")

Complete


In [2]:
# FACE RECOGNIZATION

cap = cv2.VideoCapture(0)
face_model = YOLO("models/yolov10m-face.pt")
face_recognizer = cv2.face.LBPHFaceRecognizer_create()
face_recognizer.read("face_recognizer.yml")

label_names = {0: "RIDOY"}
max_distance = 80

cv2.namedWindow("RESULT", cv2.WINDOW_NORMAL)
cv2.setWindowProperty(
    "RESULT",
    cv2.WND_PROP_FULLSCREEN,
    cv2.WINDOW_FULLSCREEN
)

while cv2.waitKey(1) != ord("x"):
    _, frame = cap.read()
    
    face_img = face_model(frame, verbose=False)
    face_result = face_img[0].plot()

    for box in face_img[0].boxes.xyxy:
        left, top, right, bottom = box.int()
        face = frame[top:bottom, left:right]
        face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
        face = cv2.resize(face, (200, 200))
    
        pred_label, distance = face_recognizer.predict(face)

        if distance > max_distance:
            name = "Unknown"
        else:
            name = label_names[pred_label]
    
        cv2.putText(
            face_result,
            name + " | " + str(int(distance)),
            (int(left) + 2, int(bottom) + 20),
            0,
            0.6,
            (255, 255, 255),
            2
        )
        
        cv2.imshow("RESULT", face_result)
    
cv2.waitKey(0)
cap.release()
cv2.destroyAllWindows()